# Metro near-duplicate finder — agi-eval-data (METRO dataset)

Finds same-map-different-bytes duplicates among the **metro/transit_dataset** maps
(85+ curated network maps). Tiny dataset (112 images) — runs on CPU or free T4.

1. Auth as your Google account
2. List files **inside the metro root folder** `1FJCnmtmeSsWfznhL0PHjYWn_btoOTRq2`
   (recursive, parents-aware — same as `scripts/metro_scan.py`)
3. Drop md5-exact copies → unique set
4. Fetch each as a ~448px Drive thumbnail (small — ~112 calls)
5. Embed with open_clip ViT-B/32 (fp32 on CPU is fine at this size)
6. Cosine > 0.95 pair search + union-find grouping
7. Write `metro-near-dup.csv` to Drive `dedup/findings/` + contact sheet

**Critical:** kept_file_id and dropped_file_id are always DIFFERENT Drive file ids.
The kept member is never listed as its own drop.

In [ ]:
# Cell 1 — auth + deps
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
drive = build('drive', 'v3')
print('AUTHORIZED-AS:', drive.about().get(fields='user').execute()['user']['emailAddress'])

!pip -q install open_clip_torch
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU mode — fine at 112 images)')

## 1 · List metro images (scoped to the metro root)

In [ ]:
# Cell 2 — list ONLY inside the metro/transit_dataset root, recursive, parents-aware
METRO_ROOT = '1FJCnmtmeSsWfznhL0PHjYWn_btoOTRq2'   # metro/transit_dataset
IMG_MIMES = ('image/jpeg','image/png','image/webp')
FIELDS = 'id,name,mimeType,md5Checksum,parents'

def list_children(folder_id):
    return drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
        pageSize=1000, fields=f'files({FIELDS})',
        supportsAllDrives=True, includeItemsFromAllDrives=True).execute().get('files', [])

# walk tree to build folder name map + collect files
folders = {}
def walk(fid, path):
    children = list_children(fid)
    for f in children:
        if f['mimeType'] == 'application/vnd.google-apps.folder':
            folders[f['id']] = (f['name'].strip(), path)
            walk(f['id'], path + '/' + f['name'].strip())
        else:
            yield f
files = list(walk(METRO_ROOT, ''))
imgs = [f for f in files if f['mimeType'] in IMG_MIMES and f.get('md5Checksum')]
print(f'{len(files)} total files · {len(imgs)} images inside metro root')
# show branch/country/city path for a few
for f in imgs[:5]:
    pid = f['parents'][0]
    print('  ', folders.get(pid, ('?', '?')), f['name'])

## 2 · Fetch thumbnails + embed (112 images — trivial)

In [ ]:
# Cell 3 — fetch thumbnails (16 threads) + CLIP embed in one pass (resumable)
import io, os, json
import requests, numpy as np
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import open_clip

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/dedup/findings'
os.makedirs(OUT_DIR, exist_ok=True)
EMB_PATH = os.path.join(OUT_DIR, 'metro_embeddings.npy')
IDS_PATH = os.path.join(OUT_DIR, 'metro_embedding_ids.json')

# md5-dedupe (keep first) AND align ids
seen, unique = set(), []
for f in imgs:
    if f['md5Checksum'] in seen: continue
    seen.add(f['md5Checksum']); unique.append(f)
ids_all = [f['id'] for f in unique]
print(f'unique images (post-md5): {len(ids_all)}')

done = set(json.load(open(IDS_PATH))) if os.path.exists(IDS_PATH) else set()
todo = [i for i in ids_all if i not in done]

S = requests.Session()
def fetch(fid):
    r = S.get(f'https://lh3.googleusercontent.com/d/{fid}=w448', timeout=30)
    if r.status_code != 200: return fid, None, r.status_code
    try: return fid, Image.open(io.BytesIO(r.content)).convert('RGB'), 200
    except Exception: return fid, None, 'decode-error'

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
model = model.eval()   # fp32; CPU fine at this size

embs  = list(np.load(EMB_PATH)) if os.path.exists(EMB_PATH) else []
ids_order = list(json.load(open(IDS_PATH))) if os.path.exists(IDS_PATH) else []
errors = []

with ThreadPoolExecutor(16) as pool:
    for chunk in [todo[i:i+32] for i in range(0, len(todo), 32)]:
        results = list(pool.map(fetch, chunk))
        ok = [(fid, img) for fid, img, _ in results if img is not None]
        errors.extend((fid, st) for fid, img, st in results if img is None)
        if not ok: continue
        batch = torch.stack([preprocess(img) for _, img in ok])
        with torch.no_grad():
            e = model(batch)
            e = e / e.norm(dim=-1, keepdim=True)
        embs.extend(e.cpu().float().numpy())
        ids_order.extend(fid for fid, _ in ok)
        np.save(EMB_PATH, np.stack(embs).astype(np.float32)); json.dump(ids_order, open(IDS_PATH,'w'))

print(f'embedded {len(ids_order)} · unreadable {len(errors)}')

## 3 · Cosine > 0.95 pairs + union-find groups

In [ ]:
# Cell 4 — union-find over cosine>0.95. X rows = global embedding indices.
# IDs are ONLY resolved through ids_order (never by raw index).
from collections import Counter

THRESH = 0.95
X  = torch.from_numpy(np.stack(embs).astype(np.float32))  # [N, 512] fp32
N  = X.shape[0]
XT = X.T

parent = list(range(N))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[max(ra, rb)] = min(ra, rb)

# blocked matmul (tiny here, but general)
B = 1024
for s in range(0, N, B):
    block = X[s:s+B] @ XT
    idx = torch.nonzero(block >= THRESH, as_tuple=False)
    for bi, j in zip(idx[:,0].tolist(), idx[:,1].tolist()):
        i = bi + s
        if j < i: union(i, j)

groups_ = {}
for i in range(N): groups_.setdefault(find(i), []).append(i)
clusters = sorted((g for g in groups_.values() if len(g) > 1), key=len, reverse=True)
print(f'{len(clusters)} near-dup clusters from {N} unique images')
print('size histogram:', dict(Counter(len(c) for c in clusters)))

## 4 · Write CSV (kept != dropped, verified)

In [ ]:
# Cell 5 — emit CSV. GUARANTEE: kept_file_id != dropped_file_id (asserted).
import csv, html

# ids_order[k] = file id of embedding row k
assert len(ids_order) == len(embs), 'ids/embs length mismatch'
gid_of = {i: ids_order[i] for i in range(len(ids_order))}
by_id  = {f['id']: f for f in unique}

rows, groups_out = [], []
for n, cluster in enumerate(clusters, 1):
    gid = f'nd-{n:05d}'
    mat = X[cluster] @ X[cluster].T
    keep_local = int(mat.mean(dim=1).argmax().item())
    keep = cluster[keep_local]
    keep_fid = gid_of[keep]
    assert all(gid_of[i] != keep_fid for i in cluster if i != keep), 'keep must differ from all drops'
    for i in cluster:
        if i == keep: continue
        cos = float(X[i] @ X[keep])
        rows.append({'group_id': gid, 'kept_file_id': keep_fid,
                     'dropped_file_id': gid_of[i], 'cosine': f'{cos:.4f}'})
    groups_out.append({'id': gid, 'kept': keep_fid, 'members': [gid_of[k] for k in cluster]})

csv_path = os.path.join(OUT_DIR, 'metro-near-dup.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['group_id','kept_file_id','dropped_file_id','cosine'])
    w.writeheader(); w.writerows(rows)
print(f'wrote {len(rows)} drop rows · {len(groups_out)} groups → {csv_path}')

# audit: every drop has a real id and differs from its keep
assert all(r['kept_file_id'] != r['dropped_file_id'] for r in rows), 'kept==dropped bug!'
print('✓ audit passed — kept always differs from dropped')

from google.colab import files as cfiles
cfiles.download(csv_path)

## 5 · Contact sheet — eyeball the flagged pairs

In [ ]:
# Cell 6 — contact sheet (all groups, side by side)
from IPython.display import HTML

parts = ['<div style="background:#111;color:#ccc;font-family:monospace;padding:8px">']
for g in groups_out:
    imgs = ''.join(
        f'<div style="text-align:center;margin:4px"><img src="https://lh3.googleusercontent.com/d/{fid}=w300" width="220"/><br>'
        f'<span style="font-size:9px">{html.escape(by_id[fid]["name"])}</span></div>'
        for fid in [g['kept']] + [i for i in g['members'] if i != g['kept']][:4])
    parts.append(f'<div style="display:flex;gap:4px;border:1px solid #333;padding:6px;margin:6px">'
                 f'<b style="min-width:90px">{g["id"]}</b>{imgs}</div>')
parts.append('</div>')
HTML(''.join(parts))